# RG Operators: Standard, Residual, Wavelet, Attention

Compares all RG operator types. Shows their Fisher metric contraction rates.



In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import torch

from src.utils.seed_registry import SeedRegistry
from src.utils.device_manager import DeviceManager

SeedRegistry.get_instance().set_master_seed(42)
device = DeviceManager.get_instance().get_device()
print(f'Device: {device}')


## Operator Implementations

In [ ]:
from src.rg_flow.operators.operators import (
    StandardRGOperator, ResidualRGOperator, WaveletRGOperator, AttentionRGOperator
)
import torch

n = 64
operators = {
    "Standard":  StandardRGOperator(n, n, sigma_w=1.4, sigma_b=0.3),
    "Residual":  ResidualRGOperator(n, n),
    "Wavelet":   WaveletRGOperator(n),
    "Attention": AttentionRGOperator(n, n_heads=4),
}

x = torch.randn(32, n)  # batch=32
print("Operator output shapes:")
for name, op in operators.items():
    out = op(x)
    print(f"  {name:<12}: {x.shape} -> {out.shape}")


## Jacobian Spectral Properties

In [ ]:
from src.core.jacobian.autograd_jacobian import AutogradJacobian

aj = AutogradJacobian()
x_single = x[0].detach().clone()

print("\nJacobian singular value statistics per operator:")
for name, op in list(operators.items())[:2]:  # first 2 for speed
    J = aj.compute(op, x_single)
    sv = torch.linalg.svdvals(J).detach().numpy()
    print(f"  {name}: max={sv.max():.4f}, min={sv.min():.4f}, mean={sv.mean():.4f}")
